In [ ]:
# Cell 1: 导入基础库并读取数据
import pandas as pd
import numpy as np
import random

# 固定全局随机种子，保证每次重启内核、Run All 后结果完全一致
SEED = 202
random.seed(SEED)
np.random.seed(SEED)

# 使用相对路径读取当前目录下的 CSV 文件
df = pd.read_csv('./heart_failure_clinical_records_dataset.csv')

# 打印数据集的行列数（形状）
print(f"数据集共有 {df.shape[0]} 行, {df.shape[1]} 列\n")

# 预览前 5 行数据
display(df.head())

In [ ]:
# Cell 2: 获取数据的统计学特征
# describe() 会自动计算所有数值列的均值、标准差和极值
stats_df = df.describe()
display(stats_df)

In [ ]:
# Cell 3: 缺失值检查与异常值箱线图
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 检查缺失值
print("=== 缺失值检查结果 ===")
missing_values = df.isnull().sum()
print(missing_values)
print("\n如果全为0，说明数据集非常干净！\n")

# 2. 画箱线图排查异常值
plt.figure(figsize=(12, 5))

# 排查射血分数
plt.subplot(1, 2, 1)
sns.boxplot(y=df['ejection_fraction'], color='skyblue')
plt.title('Boxplot: Ejection Fraction')

# 排查血清肌酐
plt.subplot(1, 2, 2)
sns.boxplot(y=df['serum_creatinine'], color='lightcoral')
plt.title('Boxplot: Serum Creatinine')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: 连续变量的标准化
from sklearn.preprocessing import StandardScaler

# 1. 明确区分连续变量和二分类变量
# 我们只对连续变量做标准化，0和1的类别变量保持原样
continuous_cols = ['age', 'creatinine_phosphokinase', 'ejection_fraction', 
                   'platelets', 'serum_creatinine', 'serum_sodium', 'time']

# 2. 初始化标准化器并拷贝一份数据
scaler = StandardScaler()
df_scaled = df.copy()

# 3. 执行标准化
df_scaled[continuous_cols] = scaler.fit_transform(df[continuous_cols])

print("=== 标准化后的前 5 行连续变量 ===")
display(df_scaled[continuous_cols].head())

In [ ]:
# Cell 5: 绘制相关性热力图
import matplotlib.pyplot as plt
import seaborn as sns

# 计算特征之间的皮尔逊相关系数
correlation_matrix = df.corr()

# 设置画布大小
plt.figure(figsize=(12, 10))

# 绘制热力图，cmap='coolwarm' 会让正相关显示为红色，负相关显示为蓝色
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', 
            linewidths=0.5, cbar_kws={"shrink": .8})

plt.title('Correlation Heatmap of Heart Failure Clinical Features', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: 使用随机森林进行特征重要性检测
from sklearn.ensemble import RandomForestClassifier

# 1. 准备数据：剔除目标变量(DEATH_EVENT)和有数据穿越嫌疑的(time)
# 注意：这里我们使用原始 df 即可，因为树模型不需要标准化
X = df.drop(['DEATH_EVENT', 'time'], axis=1)
y = df['DEATH_EVENT']

# 2. 训练一个随机森林模型
# random_state 保证每次跑出来的结果一致
# 注意：变量名改为 rf_fi_model（feature importance 专用），
# 避免和 Cell 8 里真正用于预测评估的 rf_model 重名、互相覆盖
rf_fi_model = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf_fi_model.fit(X, y)

# 3. 提取特征重要性并排序
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_fi_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# 4. 绘制特征重要性条形图
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importances, palette='viridis')
plt.title('Feature Importance for Predicting Heart Failure Mortality (Excluding Time)')
plt.xlabel('Importance Score')
plt.ylabel('Clinical Features')
plt.tight_layout()
plt.show()

# 5. 打印排名前三的危险因子
print("=== 导致心衰死亡的排名前三危险因子 ===")
display(feature_importances.head(3))

In [ ]:
# Cell 7: 划分训练集和测试集
from sklearn.model_selection import train_test_split

# 1. 准备特征矩阵 X 和目标向量 y
# 注意：使用标准化后的数据 df_scaled，并且坚决剔除卧底指标 'time'
X = df_scaled.drop(['DEATH_EVENT', 'time'], axis=1)
y = df['DEATH_EVENT']

# 2. 按 8:2 划分数据集
# stratify=y 是一个小细节，确保测试集里的死亡/存活比例和总数据一致，防止考试题全出偏了
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"训练集大小: {X_train.shape[0]} 个患者")
print(f"测试集大小: {X_test.shape[0]} 个患者")

In [ ]:
# Cell 8: 模型训练与评估 (加入类别权重平衡版)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. 初始化两个模型 【核心修复：增加 class_weight='balanced'】
# 赋予少数类（死亡高危）更高的关注度，宁可增加假阳性也要尽量避免漏诊
lr_model = LogisticRegression(class_weight='balanced', random_state=SEED)
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample', random_state=SEED)

# 2. 让模型在训练集上学习
lr_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

# 3. 让模型在测试集上做题（输出预测标签 0 或 1）
lr_preds = lr_model.predict(X_test)
rf_preds = rf_model.predict(X_test)

# 4. 定义一个评估函数来打印成绩单
def print_metrics(model_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"--- {model_name} 成绩单 ---")
    print(f"Accuracy (准确率):  {acc:.4f}")
    print(f"Precision(精确率):  {prec:.4f}")
    print(f"Recall   (召回率):  {rec:.4f}")
    print(f"F1 Score (综合分):  {f1:.4f}\n")

# 5. 打印结果
print_metrics("Logistic Regression (逻辑回归)", y_test, lr_preds)
print_metrics("Random Forest (随机森林)", y_test, rf_preds)

In [ ]:
# Cell 9: 绘制 ROC 曲线并计算 AUC
from sklearn.metrics import roc_curve, auc

# 获取模型输出的“概率值” (只取预测为 1，即死亡的概率)
lr_probs = lr_model.predict_proba(X_test)[:, 1]
rf_probs = rf_model.predict_proba(X_test)[:, 1]

# 计算假阳性率 (fpr) 和真阳性率 (tpr)
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
auc_lr = auc(fpr_lr, tpr_lr)

fpr_rf, tpr_rf, thresholds_rf = roc_curve(y_test, rf_probs)
auc_rf = auc(fpr_rf, tpr_rf)

# 开始画图
plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, color='blue', lw=2, label=f'Logistic Regression (AUC = {auc_lr:.3f})')
plt.plot(fpr_rf, tpr_rf, color='darkorange', lw=2, label=f'Random Forest (AUC = {auc_rf:.3f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Guess') # 瞎蒙基准线

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (误诊率)')
plt.ylabel('True Positive Rate (查全率 / Recall)')
plt.title('ROC Curve for Heart Failure Mortality Prediction')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Cell 10: 预测新患者的死亡概率 (修复版 + RF 对照)
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. 虚构一个极其危险的“新患者”
# 特征顺序与训练模型时的 X 保持完全一致
new_patient_data = pd.DataFrame([{
    'age': 60,                   # 样本平均年龄
    'anaemia': 0,                # 无贫血
    'creatinine_phosphokinase': 250,
    'diabetes': 0, 
    'ejection_fraction': 25,     # 射血分数极低 
    'high_blood_pressure': 0,    # 无高血压
    'platelets': 263000, 
    'serum_creatinine': 3.8,     # 肌酐偏高 
    'serum_sodium': 137,         # 平均钠
    'sex': 1, 
    'smoking': 0                 # 不抽烟
}])

print("=== 新患者原始指标 ===")
display(new_patient_data)

# 2. 【核心修复】：为这 6 个不含 time 的连续变量创建一个专属的 scaler
continuous_cols_no_time = ['age', 'creatinine_phosphokinase', 'ejection_fraction', 
                           'platelets', 'serum_creatinine', 'serum_sodium']

# 用大部队的原始数据(df)来拟合这个专属 scaler，保证均值和标准差的参照系正确
scaler_no_time = StandardScaler()
scaler_no_time.fit(df[continuous_cols_no_time]) 

# 对新病人的这 6 个连续变量进行精准标准化
new_patient_scaled = new_patient_data.copy()
new_patient_scaled[continuous_cols_no_time] = scaler_no_time.transform(new_patient_data[continuous_cols_no_time])

# 3. 输出概率预测（Logistic Regression）
# 取 lr_model 输出概率的第二个值 [1]，代表判定为“死亡”的概率
lr_death_prob = lr_model.predict_proba(new_patient_scaled)[0][1]

# 3b. 输出概率预测（Random Forest）—— 用同一份标准化后的新病人数据
rf_death_prob = rf_model.predict_proba(new_patient_scaled)[0][1]

print("\n=== AI 辅助预测结果 ===")
print(f"[Logistic Regression] 预测其在随访期内的死亡概率为: {lr_death_prob * 100:.2f}%")
print(f"[Random Forest      ] 预测其在随访期内的死亡概率为: {rf_death_prob * 100:.2f}%\n")

if lr_death_prob > 0.5:
    print(" [LR] 极高危预警：建议立即转入 ICU 重点监护")
if rf_death_prob > 0.5:
    print(" [RF] 极高危预警：建议立即转入 ICU 重点监护")
